# Debug our own model

Diagnostics only -- no training here. Loads a checkpoint written by
`train_own_model.ipynb` and asks *why* val mAP is low, one head branch at a time:

- **threshold**: is mAP being truncated by the confidence cutoff?
- **class-agnostic mAP**: are boxes + objectness fine, with classification the only blocker?
- **`branch_stats`**: objectness / classification / box quality read directly at the
  ground-truth cells, on val *and* train (overfitting vs. not learning).

Reuses the exact split saved by `train_own_model.ipynb` (`manifest.json` on Drive),
so the numbers are comparable to the training log.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

RUN_ROOT = "/content/drive/MyDrive/yolo-edge-quantization"
runDir = f"{RUN_ROOT}/own_model"

## 2. Download the dataset

The saved manifest stores absolute image paths, so the images have to exist at the
same location as during training -- same Roboflow download, same `/content` cwd.

In [ ]:
!pip install -q roboflow

from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
project = rf.workspace("efe-tatar").project("yolo-to-coco-traffic-signs-pmo2q")
dataset = project.version(1).download("coco")

datasetDirectory = dataset.location
print(datasetDirectory)

## 3. Clone (or update) the repo

In [ ]:
import os
import sys

repoDirectory = "/content/yolo-edge-quantization"
if not os.path.isdir(repoDirectory):
    !git clone -q https://github.com/efe-tatar/yolo-edge-quantization.git {repoDirectory}
else:
    !git -C {repoDirectory} pull -q

!pip install -q -r {repoDirectory}/requirements.txt
!git -C {repoDirectory} log --oneline -1

sys.path.insert(0, repoDirectory)

## 4. Load the saved split

In [ ]:
import json

with open(f"{runDir}/manifest.json") as f:
    manifest = json.load(f)
with open(f"{runDir}/classes.json") as f:
    classNames = json.load(f)
numClasses = len(classNames)

for setName in ["train", "valid", "test"]:
    print(setName, ":", len(manifest[setName]), "images")

missing = [s["image_path"] for s in manifest["valid"] + manifest["train"] if not os.path.exists(s["image_path"])]
assert not missing, f"{len(missing)} manifest images not found, e.g. {missing[0]} -- dataset path differs from training"

## 5. Load the checkpoint

`latest.pt`, not `best.pt`: `best.pt` is only rewritten when val mAP improves, and
with the old 0.3 eval threshold mAP was stuck at 0.0, so `best.pt` may still be epoch 0.
These settings must match what `train_model.py` was run with.

In [ ]:
import torch
from torch.utils.data import DataLoader

from data.dataset import DetectionDataset, collate_fn
from models.yolo import YOLO
from train.trainer import load_checkpoint, evaluate_map
from train.diagnostics import branch_stats

device = "cuda" if torch.cuda.is_available() else "cpu"
imgSize = 320
strides = (8, 16, 32)
thresholds = (32, 80)
checkpointPath = f"{runDir}/latest.pt"

model = YOLO(numClasses).to(device)
state = load_checkpoint(checkpointPath, model, device=device)
model.eval()
print("checkpoint:", checkpointPath, "| epoch:", state["epoch"], "| device:", device)

valLoader = DataLoader(DetectionDataset(manifest["valid"], img_size=imgSize),
                       batch_size=16, shuffle=False, collate_fn=collate_fn)
# a fixed train subset, enough to compare against val without a full pass
trainLoader = DataLoader(DetectionDataset(manifest["train"][:1000], img_size=imgSize),
                         batch_size=16, shuffle=False, collate_fn=collate_fn)

## 6. mAP vs. confidence threshold, normal and class-agnostic

- mAP rising as the threshold drops -> the cutoff was hiding true positives.
- class-agnostic >> normal -> boxes/objectness work, classification is what's costing mAP.
- class-agnostic also ~0 -> look at the `box_iou` columns below.

In [ ]:
import pandas as pd

rows = []
for t in (0.3, 0.01, 0.001):
    rows.append({
        "conf_thresh": t,
        "mAP@0.5": evaluate_map(model, valLoader, device, strides, numClasses, conf_thresh=t)[0.5],
        "class-agnostic mAP@0.5": evaluate_map(model, valLoader, device, strides, numClasses,
                                               conf_thresh=t, class_agnostic=True)[0.5],
    })
pd.DataFrame(rows).set_index("conf_thresh").round(4)

## 7. Per-branch stats, val vs. train

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)

statsByName = {
    "val": branch_stats(model, valLoader, device, imgSize, strides, numClasses, thresholds),
    "train[:1000]": branch_stats(model, trainLoader, device, imgSize, strides, numClasses, thresholds),
}

table = pd.DataFrame({
    (name, level): s for name, stats in statsByName.items() for level, s in stats.items()
}).T
table.round(4)

## 8. The questions that pick the fix

Read off the `all` rows:

| check | compare | meaning |
|---|---|---|
| is the classifier just guessing the most common class? | `cls_top1_acc` vs `majority_share`, plus `pred_mode_share` | top-1 ~ majority share and a high mode share = collapse onto one class |
| overfitting or not learning? | train `cls_top1_acc` vs val `cls_top1_acc` | big gap -> overfitting (augment, no h-flips); both low -> too little signal (center sampling) |
| are boxes OK? | `box_iou_median`, `box_iou>=0.5` | low -> localization is a problem too, independent of class |
| is objectness OK? | `neg_above_pos_median` | ~0 = ranks objects above background; values may still be low (`obj_pos_median`) |

In [ ]:
for name, stats in statsByName.items():
    s = stats["all"]
    print(f"--- {name} ---")
    print(f"top-1 acc            {s['cls_top1_acc']:.3f}   (chance {1/numClasses:.3f})")
    print(f"majority-class share {s['majority_share']:.3f}   class: {classNames[s['majority_class']]}")
    print(f"most-predicted share {s['pred_mode_share']:.3f}   class: {classNames[s['pred_mode_class']]}"
          f"   ({s['n_distinct_pred']} distinct classes predicted)")
    print(f"box IoU median       {s['box_iou_median']:.3f}   (>=0.5 for {s['box_iou>=0.5']:.1%} of objects)")
    print(f"obj: pos median {s['obj_pos_median']:.3f} vs neg mean {s['obj_neg_mean']:.4f},"
          f" negatives above pos median {s['neg_above_pos_median']:.2%}")
    print()